In [0]:
from pyspark.sql import SparkSession, functions as F 

from pyspark.sql.types import StringType 

  

spark = SparkSession.builder.appName("day4-assignment").getOrCreate() 

  

orders = (spark.read.option("header", True) 

          .schema("""order_id INT, order_date DATE, customer_id INT, 

                     order_status STRING, amount DOUBLE""") 

          .csv("/Volumes/yaswanth/default/data/datasets/datasets/orders.csv")) 

orders.createOrReplaceTempView("orders") 

  

customers = (spark.read.option("header", True) 

             .schema("""customer_id INT, first_name STRING, last_name STRING, 

                        city STRING, state STRING, signup_date DATE""") 

             .csv("/Volumes/yaswanth/default/data/datasets/datasets/customers.csv")) 

  

employees = spark.read.schema(""" 

    emp_id INT, name STRING, dept STRING, salary LONG, manager_id INT, 

    address STRUCT<city: STRING, state: STRING, pincode: STRING>, 

    skills ARRAY<STRING>, certifications MAP<STRING, INT>, join_date DATE 

""").json("/Volumes/yaswanth/default/data/datasets/datasets/employees.json")

In [0]:
# TASK-1
# Count orders whose status is CLOSED
closed_count = orders.filter(F.col("order_status") == "CLOSED").count()

print(closed_count)

# Count CLOSED orders using Spark SQL
spark.sql("""
    SELECT COUNT(*) AS closed_count
    FROM orders
    WHERE order_status = 'CLOSED'
""").show()

51
+------------+
|closed_count|
+------------+
|          51|
+------------+



In [0]:
#TASK-2
# Add order_month, then calculate monthly order count and total revenue
from pyspark.sql import functions as F

monthly_summary = (
    orders
    .withColumn("order_month", F.month(F.col("order_date")))
    .groupBy("order_month")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.sum("amount"), 2).alias("total_revenue")
    )
    .orderBy("order_month")
)

monthly_summary.show()


+-----------+------------+-------------+
|order_month|total_orders|total_revenue|
+-----------+------------+-------------+
|          7|         149|   2013549.66|
|          8|         151|   2008531.78|
+-----------+------------+-------------+



In [0]:
# TASK-3
# Join orders with customers and calculate total revenue per state
state_revenue = (
    orders
    .join(customers, on="customer_id", how="inner")
    .groupBy("state")
    .agg(
        F.round(F.sum("amount"), 2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

state_revenue.show()


+-----+-------------+
|state|total_revenue|
+-----+-------------+
|   KA|    994332.35|
|   MH|    772012.78|
|   TN|    771207.55|
|   WB|    409333.83|
|   AP|    370928.01|
|   DL|    359834.49|
|   TG|    248685.11|
+-----+-------------+



In [0]:
# TASK-4
# Find customers whose total spending is greater than 100000
premium_customers = (
    orders
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total_spent"))
    .filter(F.col("total_spent") > 100000)
)

premium_customers.count()


12

In [0]:
# TASK-5
# 1)A Dataset in Scala Spark provides compile-time type safety using JVM classes.
# 2)PySpark uses DataFrames, which are distributed collections of rows with a schema.
# 3)Python does not use the same JVM-based compile-time type checking as Scala, so PySpark provides DataFrames rather than Scala-style typed Datasets.
# 4)A typo like df.amout is generally caught at runtime when Spark tries to resolve the missing column, not at Python compile time.

In [0]:
# TASK-6
# Create email, initials, city_upper, and name_len columns
customers.select(
    "customer_id",
    F.lower(
        F.concat_ws(".", F.col("first_name"), F.col("last_name"))
    ).alias("full_name_temp"),
    F.concat(
        F.lower(F.col("first_name")),
        F.lit("."),
        F.lower(F.col("last_name")),
        F.lit("@shop.com")
    ).alias("email"),
    
    
    F.concat(
        F.substring(F.col("first_name"), 1, 1),
        F.substring(F.col("last_name"), 1, 1)
    ).alias("initials"),
   
   
    F.upper(F.col("city")).alias("city_upper"),
   
   
    F.length(F.col("first_name")).alias("name_len")
).drop("full_name_temp").show(5)


+-----------+--------------------+--------+----------+--------+
|customer_id|               email|initials|city_upper|name_len|
+-----------+--------------------+--------+----------+--------+
|          1| aarav.nair@shop.com|      AN| BENGALURU|       5|
|          2|vikram.reddy@shop...|      VR|    MUMBAI|       6|
|          3|  varun.rao@shop.com|      VR| BENGALURU|       5|
|          4|rohan.patel@shop.com|      RP| HYDERABAD|       5|
|          5|neha.sharma@shop.com|      NS|    MYSURU|       4|
+-----------+--------------------+--------+----------+--------+
only showing top 5 rows


In [0]:
# TASK-7
# Calculate GST-inclusive amount and amount in thousands
orders.select(
    "order_id",
    "amount",
    F.round(F.col("amount") * 1.18, 2).alias("amount_with_gst"),
    F.floor(F.col("amount") / 1000).alias("amount_k")
).show(5)


+--------+--------+---------------+--------+
|order_id|  amount|amount_with_gst|amount_k|
+--------+--------+---------------+--------+
|       1| 9355.49|       11039.48|       9|
|       2| 6633.45|        7827.47|       6|
|       3|  6646.6|        7842.99|       6|
|       4| 2064.02|        2435.54|       2|
|       5|22397.18|       26428.67|      22|
+--------+--------+---------------+--------+
only showing top 5 rows


In [0]:
#TASK-8
# Add date columns, count orders by weekday, and find min/max day differences
orders_dates = (
    orders
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("day_name", F.date_format("order_date", "EEEE"))
    .withColumn(
        "days_to_oct1",
        F.datediff(
            F.lit("2026-10-01").cast("date"),
            F.col("order_date")
        )
    )
)

# Count orders per day name, busiest first
orders_dates.groupBy("day_name").count().orderBy(
    F.col("count").desc()
).show()

# Print minimum and maximum days_to_oct1
orders_dates.agg(
    F.min("days_to_oct1").alias("min_days_to_oct1"),
    F.max("days_to_oct1").alias("max_days_to_oct1")
).show()


+---------+-----+
| day_name|count|
+---------+-----+
| Thursday|   45|
|   Monday|   44|
|   Sunday|   43|
|   Friday|   43|
| Saturday|   42|
|  Tuesday|   42|
|Wednesday|   41|
+---------+-----+

+----------------+----------------+
|min_days_to_oct1|max_days_to_oct1|
+----------------+----------------+
|              32|              92|
+----------------+----------------+



In [0]:
# TASK-9
# Classify orders into amount bands and count each band
orders_bands = orders.withColumn(
    "amount_band",
    F.when(F.col("amount").isNull(), "UNKNOWN")
     .when(F.col("amount") >= 20000, "HIGH")
     .when(F.col("amount") >= 5000, "MEDIUM")
     .otherwise("LOW")
)

orders_bands.groupBy("amount_band").count().show()


+-----------+-----+
|amount_band|count|
+-----------+-----+
|     MEDIUM|  169|
|        LOW|   52|
|    UNKNOWN|   10|
|       HIGH|   69|
+-----------+-----+



In [0]:
#TASK-10


# Count orders where amount is null
orders.filter(F.col("amount").isNull()).count()


# Replace null amount with 0 using coalesce
orders.filter(F.col("amount").isNull()).select(
    "order_id",
    F.coalesce(F.col("amount"), F.lit(0.0)).alias("amount")
).show()



# Count rows after removing records with null amount
orders.dropna(subset=["amount"]).count()



# Compare total amount before and after filling nulls with 0
orders.agg(F.sum("amount").alias("revenue_before")).show()

orders.fillna({"amount": 0}).agg(
    F.sum("amount").alias("revenue_after")
).show()


+--------+------+
|order_id|amount|
+--------+------+
|      21|   0.0|
|      27|   0.0|
|      48|   0.0|
|      62|   0.0|
|      83|   0.0|
|     113|   0.0|
|     179|   0.0|
|     277|   0.0|
|     289|   0.0|
|     297|   0.0|
+--------+------+

+------------------+
|    revenue_before|
+------------------+
|4022081.4399999976|
+------------------+

+------------------+
|     revenue_after|
+------------------+
|4022081.4399999976|
+------------------+



In [0]:
#TASK-11
# Calculate order statistics for every customer and show the top 5 by revenue
customer_summary = (
    orders
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("number_of_orders"),
        F.round(F.sum("amount"), 2).alias("total_revenue"),
        F.round(F.avg("amount"), 2).alias("average_amount"),
        F.round(F.max("amount"), 2).alias("largest_order"),
        F.countDistinct("order_status").alias("distinct_status_count"),
        F.collect_set("order_status").alias("statuses")
    )
    .orderBy(F.col("total_revenue").desc())
)

customer_summary.show(5, truncate=False)


+-----------+----------------+-------------+--------------+-------------+---------------------+------------------------------------------------------------------+
|customer_id|number_of_orders|total_revenue|average_amount|largest_order|distinct_status_count|statuses                                                          |
+-----------+----------------+-------------+--------------+-------------+---------------------+------------------------------------------------------------------+
|24         |10              |161189.59    |16118.96      |33409.65     |3                    |[CLOSED, ON_HOLD, COMPLETE]                                       |
|30         |10              |152688.93    |15268.89      |23835.35     |6                    |[CLOSED, COMPLETE, CANCELED, SUSPECTED_FRAUD, PROCESSING, PENDING]|
|11         |9               |149412.14    |16601.35      |39998.5      |4                    |[PENDING, PENDING_PAYMENT, CLOSED, COMPLETE]                      |
|5          |9        